# 13 End To End Multi Agent System

In [98]:
from typing import TypedDict
from typing import List
from typing import Dict
from typing import Annotated

import operator

from langgraph.graph import StateGraph, END

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from transformers import pipeline

In [99]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [100]:
llm = pipeline(
    "text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=512
)

Device set to use cpu


In [101]:
class EnterpriseResearchState(TypedDict):

    research_goal: str

    findings: Annotated[List[str], operator.add]

    missing_topics: List[str]

    conflicts: List[Dict]

    trust_score: float

    critic_score: float

    compressed_memory: str

    approval_status: str

    final_report: str

    iteration_count: int

    status: str

    metadata: Dict

In [102]:
def research_agent(state):

    print("\nResearch Agent")

    findings = [
        "AWS pricing increased 8%",
        "Azure pricing remained stable",
        "Google Cloud storage costs reduced 4%",
        "Vendor lock-in risk identified",
    ]

    return {"findings": findings, "status": "research_completed"}

In [103]:
def deduplication_agent(state):

    print("\nDeduplication Agent")

    unique_findings = list(set(state["findings"]))

    return {"findings": unique_findings}

In [104]:
def critic_agent(state):

    print("\nCritic Agent")

    goal = state["research_goal"]

    findings = " ".join(state["findings"])

    goal_emb = embedding_model.encode(goal)

    findings_emb = embedding_model.encode(findings)

    score = cosine_similarity(goal_emb.reshape(1, -1), findings_emb.reshape(1, -1))[0][
        0
    ]

    return {"critic_score": float(score)}

In [105]:
EXPECTED_TOPICS = ["AWS", "Azure", "Google Cloud", "Oracle Cloud"]

In [106]:
def gap_analysis_agent(state):

    print("\nGap Analysis Agent")

    text = " ".join(state["findings"]).lower()

    missing = []

    for topic in EXPECTED_TOPICS:

        if topic.lower() not in text:

            missing.append(topic)

    return {"missing_topics": missing}

In [107]:
def fact_checker_agent(state):

    print("\nFact Checker Agent")

    conflicts = []

    state["conflicts"] = conflicts

    return {"conflicts": [], "trust_score": 0.95}

In [108]:
def human_review_agent(state):

    return {
        "approval_status": "approved" if state["trust_score"] >= 0.80 else "rejected"
    }

In [109]:
def approval_router(state):

    if state["approval_status"] == "approved":

        return "finance"

    return END

In [110]:
def finance_agent(state):

    print("\nFinance Agent")

    return {"findings": ["Cloud ROI improved 7%"]}

In [111]:
def competitor_agent(state):

    print("\nCompetitor Agent")

    return {"findings": ["Oracle Cloud expanding AI GPUs"]}

In [112]:
def risk_agent(state):

    print("\nRisk Agent")

    return {"findings": ["Migration risk remains high"]}

In [113]:
def compliance_agent(state):

    print("\nCompliance Agent")

    return {"findings": ["GDPR review required"]}

In [114]:
def merge_agent(state):

    print("\nMerge Agent")

    return {}

In [115]:
def memory_agent(state):

    print("\nMemory Agent")

    return {"compressed_memory": "Cloud market remains highly competitive."}

In [116]:
def writer_agent(state):

    print("\nWriter Agent")

    findings = "\n".join(state["findings"])

    prompt = f"""
Create an executive report.

Goal:
{state['research_goal']}

Findings:
{findings}
"""

    result = llm(prompt)

    return {"final_report": result[0]["generated_text"], "status": "completed"}

In [117]:
graph = StateGraph(EnterpriseResearchState)

In [118]:
graph.add_node("research", research_agent)

graph.add_node("deduplicate", deduplication_agent)

graph.add_node("critic", critic_agent)

graph.add_node("gap", gap_analysis_agent)

graph.add_node("factcheck", fact_checker_agent)

graph.add_node("review", human_review_agent)

graph.add_node("finance", finance_agent)

graph.add_node("competitor", competitor_agent)

graph.add_node("risk", risk_agent)

graph.add_node("compliance", compliance_agent)

graph.add_node("merge", merge_agent)

graph.add_node("memory", memory_agent)

graph.add_node("writer", writer_agent)

In [119]:
graph.set_entry_point("research")

In [120]:
graph.add_edge("research", "deduplicate")

graph.add_edge("deduplicate", "critic")

graph.add_edge("critic", "gap")

graph.add_edge("gap", "factcheck")

graph.add_edge("factcheck", "review")

In [121]:
graph.add_conditional_edges("review", approval_router)

In [122]:
graph.add_edge("finance", "competitor")
graph.add_edge("competitor", "risk")
graph.add_edge("risk", "compliance")
graph.add_edge("compliance", "merge")

In [123]:
graph.add_edge("merge", "memory")

graph.add_edge("memory", "writer")

graph.add_edge("writer", END)

In [124]:
app = graph.compile()

In [125]:
initial_state = {
    "research_goal": """
    Compare cloud infrastructure
    costs against competitors
    and identify risks
    """,
    "findings": [],
    "missing_topics": [],
    "conflicts": [],
    "trust_score": 0.0,
    "critic_score": 0.0,
    "compressed_memory": "",
    "approval_status": "",
    "final_report": "",
    "iteration_count": 0,
    "status": "",
    "metadata": {},
}

In [126]:
result = app.invoke(initial_state)


Research Agent

Deduplication Agent

Critic Agent

Gap Analysis Agent

Fact Checker Agent

Finance Agent

Competitor Agent

Risk Agent

Compliance Agent

Merge Agent

Memory Agent

Writer Agent


In [127]:
print(result["final_report"])


Create an executive report.

Goal:

    Compare cloud infrastructure
    costs against competitors
    and identify risks
    

Findings:
AWS pricing increased 8%
Azure pricing remained stable
Google Cloud storage costs reduced 4%
Vendor lock-in risk identified
Vendor lock-in risk identified
Google Cloud storage costs reduced 4%
AWS pricing increased 8%
Azure pricing remained stable
Cloud ROI improved 7%
Oracle Cloud expanding AI GPUs
Migration risk remains high
GDPR review required
Data security concerns persist

Recommendations:
1. Conduct a thorough comparison of AWS, Azure, Google Cloud,
pricing, features, and customer reviews.
2. Assess the vendor lock-in risk by evaluating contracts with each provider.
3. Explore alternative cloud providers to reduce costs or improve ROI.
4. Develop a plan for migrating Oracle Cloud instances to AWS or Azure.
5. Implement data encryption and access controls to address GDPR requirements.
6. Enhance data security measures to protect sensitive info